# Fase 2 Minería de Datos NYC311

## 1 Qué vamos a hacer

Este notebook continúa el trabajo de la Fase 1. No toma los datos directamente de la capa RAW: consulta la vista analítica `mining.vw_dataset_fase2` de `DW_NYC311` como fuente única del análisis. La consulta no aplica un filtro temporal: `period_label` y la versión del modelo se derivan del rango realmente disponible en la vista.

Seguiremos el proceso KDD. Primero revisamos y preparamos los datos; después aplicamos un Árbol de Decisión para clasificar demora alta y K-Means para agrupar perfiles operativos. Finalmente evaluamos las métricas y traducimos los resultados a acciones posibles.

La cadena de trazabilidad es: NYC Open Data → ETL Fase 1 → `raw.Solicitud` → `stg.Solicitud` → `DW_NYC311` → `mining.vw_dataset_fase2` → pandas y scikit-learn.


## 2 Cargar el dataset

`pandas` es una biblioteca de Python para trabajar con tablas. Al consultar SQL Server obtenemos un `DataFrame`, que puede entenderse como una tabla con filas y columnas.

La fuente es SQL Server mediante autenticación Windows y el controlador ODBC de Microsoft. Se consulta todo el rango disponible de la vista; las credenciales SQL, si se necesitan, se leen de `DB_USER` y `DB_PASSWORD`.

Al final se muestra `df.shape`, que responde cuántas filas y columnas se cargaron, y `df.head()`, que permite revisar algunos registros sin modificar los datos. La celda también valida columnas, unicidad de solicitudes y valores sin duración.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, silhouette_score,
)
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid", context="notebook")
SQL_SERVER = "localhost"
SQL_DATABASE = "DW_NYC311"
EXPECTED_COLUMNS = [
    "codigo_solicitud", "agency", "complaint_type", "borough", "status",
    "open_data_channel_type", "anio", "mes", "dia_semana",
    "minutos_resolucion", "demora_alta", "latitude", "longitude",
]
available_drivers = pyodbc.drivers()
driver = next((d for d in ("ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server") if d in available_drivers), None)
if driver is None:
    raise RuntimeError("No se encontró ODBC Driver 18/17 para SQL Server.")
connection_options = {
    "driver": driver,
    "TrustServerCertificate": "yes",
    "Encrypt": "no",
}
credentials = {}
if os.getenv("DB_USER") and os.getenv("DB_PASSWORD"):
    credentials = {"username": os.environ["DB_USER"], "password": os.environ["DB_PASSWORD"]}
else:
    connection_options["Trusted_Connection"] = "yes"
connection_url = URL.create(
    "mssql+pyodbc", host=SQL_SERVER, database=SQL_DATABASE,
    query=connection_options, **credentials
)
query = text("""
    SELECT codigo_solicitud, agency, complaint_type, borough, status,
           open_data_channel_type, anio, mes, dia_semana,
           minutos_resolucion, demora_alta, latitude, longitude
    FROM mining.vw_dataset_fase2;
""")
engine = create_engine(connection_url)
with engine.connect() as conn:
    df = pd.read_sql_query(
        query,
        conn,
    )
engine.dispose()
if df.empty:
    raise ValueError("mining.vw_dataset_fase2 no devolvió filas.")
period_start = pd.Timestamp(
    year=int(df["anio"].min()), month=int(df.loc[df["anio"] == df["anio"].min(), "mes"].min()), day=1
).date()
period_end = (
    pd.Timestamp(year=int(df["anio"].max()), month=int(df.loc[df["anio"] == df["anio"].max(), "mes"].max()), day=1)
    + pd.offsets.MonthEnd(0)
).date()
period_label = f"{period_start:%Y-%m-%d} a {period_end:%Y-%m-%d}"
MODEL_VERSION = f"F2-notebook-K3-r42-n10-{period_start:%Y%m%d}_{period_end:%Y%m%d}"
source_label = f"SQL Server {SQL_SERVER}/{SQL_DATABASE}.mining.vw_dataset_fase2 ({period_label})"

missing_columns = [column for column in EXPECTED_COLUMNS if column not in df.columns]
if missing_columns:
    raise ValueError(f"La fuente no cumple el contrato Fase 2. Faltan: {missing_columns}")
if df["codigo_solicitud"].duplicated().any():
    raise ValueError("La vista no conserva el grano esperado: hay solicitudes duplicadas.")
null_duration = df["minutos_resolucion"].isna().sum()
print(f"Período disponible: {period_label}")
print(f"Fuente única: {source_label}")
print(f"Dimensiones: {df.shape}")
print(f"Solicitudes sin duración: {null_duration:,}")
display(df.head())

## 3 Entender el dataset

Antes de modelar comprobamos el contrato de columnas de la vista de minería. `df.shape` muestra el tamaño del dataset, `df.dtypes` muestra el tipo de cada columna y `df.info()` permite ver rápidamente tipos y valores no nulos cuando se ejecuta de forma interactiva.

También revisamos nulos, duplicados y cardinalidad. La cardinalidad indica cuántos valores distintos tiene una categoría, por ejemplo cuántas agencias o tipos de reclamo aparecen. Esto ayuda a anticipar cómo se transformarán las columnas categóricas.


In [ ]:
required_columns = [
    "codigo_solicitud", "agency", "complaint_type", "borough", "status",
    "open_data_channel_type", "anio", "mes", "dia_semana",
    "minutos_resolucion", "demora_alta", "latitude", "longitude",
]
missing_columns = sorted(set(required_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(required_columns))
if missing_columns or unexpected_columns:
    raise ValueError(
        f"Contrato de columnas inválido. Faltantes={missing_columns}; "
        f"inesperadas={unexpected_columns}"
    )
df = df[required_columns].copy()
print("Tipos de datos:")
df.info()
display(df.dtypes.to_frame("dtype"))
print("Valores nulos:")
display(df.isna().sum().to_frame("nulos").query("nulos > 0"))
print(f"Duplicados completos: {df.duplicated().sum():,}")
print("Cardinalidad de variables categóricas:")
display(df[["agency", "complaint_type", "borough", "status", "open_data_channel_type"]].nunique())

## 4 Revisar calidad de datos

Un valor nulo no siempre es un error. En este dataset, `minutos_resolucion` puede estar vacío cuando una solicitud todavía no tiene cierre registrado. No lo reemplazamos por un promedio porque eso inventaría un tiempo que no se ha observado.

En la siguiente celda contamos esos casos por estado y comprobamos que no existan duplicados completos. Las solicitudes sin tiempo final se conservan para describir el rezago, pero no se usan para entrenar la clasificación de demora alta.

## 5 Construir demora alta

Un percentil indica la posición de un valor dentro de una distribución. El percentil 75, o P75, deja aproximadamente 75% de los tiempos observados por debajo o igual al corte y 25% por encima.

NYC Open Data no entrega una etiqueta llamada `demora_alta`. En este notebook conservamos la etiqueta que entrega la vista `mining.vw_dataset_fase2` y validamos su umbral observado; el P75 recalculado sobre todo el rango se muestra como diagnóstico, pero no reemplaza la regla F2 persistida. El umbral no es un estándar universal ni vino de NYC Open Data, scikit-learn o el docente.

`minutos_resolucion` es una variable continua. `demora_alta` la discretiza en dos clases: 0 para tiempo menor o igual al P75 y 1 para tiempo superior. Los valores sin cierre quedan sin clasificar.


In [ ]:
null_resolution_by_status = (
    df.loc[df["minutos_resolucion"].isna(), "status"]
    .value_counts(dropna=False)
    .rename("solicitudes_sin_resolucion")
)
display(null_resolution_by_status.to_frame())

resolved = df["minutos_resolucion"].dropna()
recomputed_p75 = float(resolved.quantile(0.75))
source_threshold = float(df.loc[df["demora_alta"].eq(0), "minutos_resolucion"].max())
source_high_minimum = float(df.loc[df["demora_alta"].eq(1), "minutos_resolucion"].min())
if not source_high_minimum > source_threshold:
    raise ValueError("La etiqueta demora_alta de la vista no tiene un umbral monotónico observable.")
p75 = source_threshold
target_recomputed = pd.Series(pd.NA, index=df.index, dtype="Int64")
has_resolution = df["minutos_resolucion"].notna()
target_recomputed.loc[has_resolution] = (
    df.loc[has_resolution, "minutos_resolucion"].gt(p75).astype("int64")
)
assert df["demora_alta"].astype("Int64").equals(target_recomputed)
df["demora_alta"] = target_recomputed
print(f"Umbral de demora_alta conservado desde la vista: {p75:,.0f} minutos")
print(f"P75 recalculado solo como diagnóstico: {recomputed_p75:,.0f} minutos")
display(df["demora_alta"].value_counts(dropna=False).sort_index().rename("solicitudes").to_frame())

Las gráficas siguientes permiten revisar visualmente la distribución del tiempo de resolución y el volumen de nulos. Se usan para entender los datos; no cambian ninguna observación.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(resolved, bins=80, color="#2f6f9f")
axes[0].axvline(p75, color="#d95f02", linestyle="--", label=f"P75 = {p75:,.0f}")
axes[0].set_xscale("symlog", linthresh=60)
axes[0].set_title("Tiempo de resolución")
axes[0].set_xlabel("Minutos escala symlog")
axes[0].legend()
nulls = df.isna().sum().sort_values(ascending=False)
nulls = nulls[nulls > 0]
axes[1].bar(nulls.index, nulls.values, color="#d95f02")
axes[1].set_title("Valores nulos")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6 Seleccionar variables

Para clasificar una solicitud usamos `agency`, `complaint_type`, `borough`, `open_data_channel_type`, `mes` y `dia_semana`. Estas variables están disponibles como contexto de la solicitud y no incluyen su tiempo final.

Se excluye `minutos_resolucion` por **data leakage**. La fuga de información ocurre cuando se entrega al modelo un dato que, en un escenario real, todavía no estaría disponible al hacer la predicción. Aquí no puede ser predictor porque `demora_alta` se construyó precisamente a partir de `minutos_resolucion`.

También quedan fuera `status`, el identificador, el año y las coordenadas. `status` puede reflejar información posterior al registro; el código no tiene significado predictivo; el año se mantiene fuera para conservar el contrato F2; y las coordenadas se reservan para otro análisis.

## 7 Separar X e y

`X` es la tabla de variables que el modelo puede observar. `y` es la respuesta que queremos predecir: `demora_alta`. Solo se incluyen filas donde esa respuesta existe; el conteo de clasificables se obtiene del rango consultado al ejecutar.

## 8 Train Test

Usamos 80% de las solicitudes clasificables para entrenamiento y 20% para prueba. Entrenamiento significa que el modelo aprende patrones con esos datos. Prueba significa que reservamos datos para verificar si el modelo generaliza a casos que no vio durante el ajuste.

`stratify=y` conserva aproximadamente la proporción 75/25 entre clase normal y demora alta en ambos grupos. `random_state=42` permite repetir exactamente la misma partición.

## 9 OneHotEncoder

El modelo necesita números, pero no debemos convertir `agency=NYPD` en un número mayor o menor que `agency=HPD`. `OneHotEncoder` crea columnas binarias: una columna indica si la agencia es NYPD y otra si es HPD. Así representa categorías sin inventar un orden numérico.

## 10 Pipeline

Un `Pipeline` une el preprocesamiento y el modelo. De esta forma, el mismo `OneHotEncoder` se aplica durante el entrenamiento y durante una predicción, sin depender de pasos manuales diferentes.

## 11 Árbol baseline

Un Árbol de Decisión toma decisiones sucesivas para separar clases. En cada nodo busca una división que deje grupos más puros. Su impureza puede medirse con Gini: si un nodo contiene casi una sola clase, su Gini se acerca a 0.

`max_depth` limita cuántos niveles puede crecer el árbol. `min_samples_leaf` evita hojas creadas con muy pocos casos. El baseline usa profundidad 6 y al menos 100 casos por hoja; será el punto de comparación para el ajuste posterior.

## 12 Evaluación baseline

Las métricas responden preguntas distintas sobre las solicitudes con demora alta. Accuracy es el porcentaje total de aciertos. Precision pregunta cuántas alertas emitidas sí eran demora alta. Recall pregunta cuántas demoras altas reales fueron detectadas. F1 combina precision y recall. ROC-AUC evalúa qué tan bien el modelo ordena el riesgo usando probabilidades.

La matriz de confusión separa verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos. En este proyecto interesan especialmente los falsos negativos: demoras reales que el modelo no alertó.

## 13 GridSearchCV

Un hiperparámetro es una configuración del modelo que se define antes de entrenar. `GridSearchCV` prueba combinaciones de hiperparámetros y elige la mejor según F1. Aquí evalúa 3 valores de profundidad, 3 mínimos por hoja y 2 pesos de clase: 18 combinaciones.

La validación cruzada divide los datos de entrenamiento en tres partes. En cada ronda entrena con dos partes y valida con la restante; luego rota la parte validada. Por eso se realizan 18 × 3 = 54 ajustes. Optimizamos F1 porque equilibra precisión y recall en una clase de demora alta menos frecuente.


In [ ]:
categorical_features = ["agency", "complaint_type", "borough", "open_data_channel_type"]
numeric_features = ["mes", "dia_semana"]
model_features = categorical_features + numeric_features
classification_df = df.loc[df["demora_alta"].notna()].copy()
X = classification_df[model_features]
y = classification_df["demora_alta"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
transformer = ColumnTransformer(
    [("categoricas", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
    remainder="passthrough",
)

def make_pipeline(class_weight=None, max_depth=None, min_samples_leaf=1):
    return Pipeline([
        ("preprocesamiento", transformer),
        ("modelo", DecisionTreeClassifier(
            class_weight=class_weight,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
        )),
    ])

baseline = make_pipeline(max_depth=6, min_samples_leaf=100)
baseline.fit(X_train, y_train)
RUN_GRID_SEARCH = True  # Use False durante una demostración corta.
if not RUN_GRID_SEARCH:
    optimized = make_pipeline(
        class_weight="balanced", max_depth=15, min_samples_leaf=50
    )
    optimized.fit(X_train, y_train)
    grid = None
    print("Modo demostración: se usan los mejores parámetros ya validados.")
else:
    grid = GridSearchCV(
        make_pipeline(),
        {
            "modelo__max_depth": [6, 10, 15],
            "modelo__min_samples_leaf": [50, 100, 250],
            "modelo__class_weight": [None, "balanced"],
        },
        scoring="f1", cv=3, n_jobs=-1, verbose=2,
    )
    grid.fit(X_train, y_train)
    optimized = grid.best_estimator_
    print("Mejores parámetros:", grid.best_params_)
    print(f"Mejor F1 de validación cruzada: {grid.best_score_:.6f}")

## 14 Árbol optimizado

El modelo optimizado usa `class_weight="balanced"`, que aumenta el peso relativo de la clase minoritaria. El trade-off observado es esperado: Recall sube para dejar menos demoras sin alerta, mientras Precision baja porque se generan más alertas preventivas. Por eso el árbol se propone como filtro para revisión humana, no como decisión automática.

La siguiente celda calcula las métricas de baseline y modelo optimizado sobre el conjunto de prueba apartado. También muestra la matriz de confusión y el reporte de clasificación.


In [ ]:
def evaluate(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    proba = model.predict_proba(X_eval)[:, 1]
    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred, zero_division=0),
        "F1": f1_score(y_eval, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, proba),
        "Matriz": confusion_matrix(y_eval, pred),
        "pred": pred,
        "proba": proba,
    }

baseline_results = evaluate(baseline, X_test, y_test)
optimized_results = evaluate(optimized, X_test, y_test)
comparison = pd.DataFrame(
    {"Baseline": {k: baseline_results[k] for k in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]},
     "Optimizado": {k: optimized_results[k] for k in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]}}
)
display(comparison.style.format("{:.6f}"))
print("Matriz optimizada:")
print(optimized_results["Matriz"])
print(classification_report(y_test, optimized_results["pred"], digits=6))

## 15 Importancia de variables

La importancia de una variable indica cuánto la utiliza este árbol para separar las clases. No demuestra que esa variable cause una demora. En particular, que `agency` tenga alta importancia predictiva no permite afirmar que una agencia sea la causa del tiempo de resolución.

La siguiente celda recupera `feature_importances_`, agrupa las columnas creadas por OneHotEncoder y muestra la matriz de confusión y la curva ROC del modelo optimizado.


In [ ]:
feature_names = optimized.named_steps["preprocesamiento"].get_feature_names_out()
importances = optimized.named_steps["modelo"].feature_importances_
importance_detail = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_detail["feature"] = importance_detail["feature"].str.split("__").str[-1]
importance_detail["source"] = importance_detail["feature"]
for column in categorical_features:
    importance_detail.loc[importance_detail["feature"].str.startswith(column + "_"), "source"] = column
importance_by_source = (
    importance_detail.groupby("source", as_index=False)["importance"]
    .sum().sort_values("importance", ascending=False)
)
display(importance_by_source.style.format({"importance": "{:.6f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(optimized_results["Matriz"], annot=True, fmt=",", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("Matriz de confusión optimizada")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Valor real")
RocCurveDisplay.from_predictions(y_test, optimized_results["proba"], ax=axes[1], name="Árbol optimizado")
axes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

## 16 Preparar K Means

K-Means no se aplica directamente a los tickets. Primero resumimos todas las solicitudes consultadas en un perfil definido por `agency + complaint_type + borough`. Cada perfil conserva cuántas solicitudes tiene, su mediana de resolución, el porcentaje de demora y el porcentaje sin resolución.

Así pasamos de muchas filas individuales a perfiles por terna. La celda imprime cuántos perfiles existen antes del filtro, cuántos cumplen al menos 100 solicitudes y cuántos conservan métricas suficientes para el clustering. Este filtro evita construir grupos a partir de combinaciones demasiado pequeñas.

## 17 log1p

`log1p` aplica el logaritmo de `1 + valor`. Se usa en el número de solicitudes y la mediana de resolución para comprimir valores muy grandes y sesgados sin cambiar el orden entre perfiles. Un perfil de gran volumen sigue siendo mayor que uno pequeño, pero deja de dominar únicamente por su escala numérica.

## 18 StandardScaler

K-Means mide distancias. `StandardScaler` transforma las variables para que sean comparables en escala. Sin esta normalización, una variable numéricamente grande, como la cantidad de solicitudes, podría dominar la distancia y ocultar el efecto de los porcentajes.


In [ ]:
profiles = (
    df.groupby(["agency", "complaint_type", "borough"], dropna=False)
    .agg(
        solicitudes=("codigo_solicitud", "count"),
        resueltas=("minutos_resolucion", "count"),
        mediana_resolucion=("minutos_resolucion", "median"),
        promedio_resolucion=("minutos_resolucion", "mean"),
        porcentaje_demora=("demora_alta", "mean"),
    ).reset_index()
)
profile_count_before = len(profiles)
profiles["porcentaje_sin_resolucion"] = 1 - profiles["resueltas"] / profiles["solicitudes"]
profiles = profiles.loc[profiles["solicitudes"] >= 100].copy()
profile_count_after = len(profiles)
profiles = profiles.dropna(subset=["mediana_resolucion", "porcentaje_demora"]).copy()
profiles["log_solicitudes"] = np.log1p(profiles["solicitudes"])
profiles["log_mediana_resolucion"] = np.log1p(profiles["mediana_resolucion"])
cluster_features = [
    "log_solicitudes", "log_mediana_resolucion",
    "porcentaje_demora", "porcentaje_sin_resolucion",
]
scaler = StandardScaler()
scaled_profiles = scaler.fit_transform(profiles[cluster_features])
print(f"Perfiles originales: {profile_count_before}")
print(f"Perfiles con mínimo 100 solicitudes: {profile_count_after}")
print(f"Perfiles finales para K-Means: {len(profiles)}")

## 19 Seleccionar K

`K` es el número de grupos que queremos buscar. Un cluster es un grupo de perfiles parecidos y su centroide es el punto promedio que representa al grupo. K-Means asigna cada perfil al centroide más cercano y actualiza esos centros hasta reducir las distancias dentro de los clusters.

La Inertia suma las distancias cuadradas de los perfiles a su centroide; valores más bajos indican grupos más compactos, aunque siempre tiende a bajar cuando se agregan más clusters. Silhouette compara qué tan parecido es un perfil a su propio grupo frente a otros grupos; valores mayores indican mejor separación.

Se prueba K entre 2 y 8 para observar Inertia y Silhouette. Se conserva K=3, la configuración definida en Fase 2; el valor obtenido para este rango se muestra en `k_metrics` al ejecutar. Es una separación exploratoria: los grupos pueden solaparse y no deben interpretarse como categorías absolutas.

## 20 K Means final

`fit_predict` ajusta K-Means y devuelve el cluster asignado a cada perfil. Después nombramos los tres clusters según sus métricas observadas para facilitar una interpretación de negocio.


In [ ]:
k_metrics = []
for k in range(2, 9):
    candidate = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = candidate.fit_predict(scaled_profiles)
    k_metrics.append({
        "K": k,
        "Inertia": candidate.inertia_,
        "Silhouette": silhouette_score(scaled_profiles, labels),
    })
k_metrics = pd.DataFrame(k_metrics)
display(k_metrics.style.format({"Inertia": "{:.6f}", "Silhouette": "{:.6f}"}))
final_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
profiles["cluster"] = final_kmeans.fit_predict(scaled_profiles)
raw_cluster_summary = profiles.groupby("cluster").agg(
    perfiles=("cluster", "size"),
    solicitudes=("solicitudes", "sum"),
    mediana_resolucion=("mediana_resolucion", "median"),
    porcentaje_demora=("porcentaje_demora", "mean"),
    porcentaje_sin_resolucion=("porcentaje_sin_resolucion", "mean"),
)
order = raw_cluster_summary.sort_values("mediana_resolucion").index.tolist()
cluster_names = {
    order[0]: "Resolución rápida y baja demora",
    order[1]: "Resolución prolongada y alta demora",
    order[2]: "Rezago crítico y alta proporción sin resolver",
}
profiles["cluster_nombre"] = profiles["cluster"].map(cluster_names)
cluster_summary = raw_cluster_summary.reset_index()
cluster_summary["cluster_nombre"] = cluster_summary["cluster"].map(cluster_names)
display(cluster_summary.style.format({
    "porcentaje_demora": "{:.2%}",
    "porcentaje_sin_resolucion": "{:.2%}",
    "mediana_resolucion": "{:.0f}",
}))

Las gráficas siguientes muestran cómo cambia Inertia y Silhouette al variar K, y dónde quedan los perfiles operativos. El tamaño de cada punto representa el volumen de solicitudes del perfil.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(k_metrics["K"], k_metrics["Inertia"], marker="o", color="#2f6f9f")
axes[0].set_title("Inertia")
axes[0].set_xlabel("K")
axes[1].plot(k_metrics["K"], k_metrics["Silhouette"], marker="o", color="#d95f02")
axes[1].set_title("Silhouette")
axes[1].set_xlabel("K")
for name, group in profiles.groupby("cluster_nombre"):
    axes[2].scatter(
        group["mediana_resolucion"], group["porcentaje_sin_resolucion"] * 100,
        s=np.clip(group["solicitudes"] / 20, 18, 170), alpha=.7, label=name,
    )
axes[2].set_xscale("symlog", linthresh=60)
axes[2].set_title("Perfiles operativos")
axes[2].set_xlabel("Mediana de resolución escala symlog")
axes[2].set_ylabel("% sin resolución")
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.show()

## 21 Interpretación

Los tres perfiles se renombran por orden ascendente de mediana de resolución: resolución rápida, resolución prolongada y rezago crítico. Las cantidades y porcentajes se muestran en `cluster_summary` y no se fijan en el texto, porque dependen del rango actualmente disponible en la vista.

El árbol ayuda a priorizar alertas de demora. K-Means ayuda a decidir dónde investigar procesos. La recomendación operativa es revisar los perfiles de alto volumen en resolución prolongada y auditar primero los estados y cierres del rezago crítico.

## 22 Conclusiones y limitaciones

El análisis identifica patrones predictivos y perfiles de operación, pero no prueba causalidad. El período se obtiene del rango realmente disponible en la vista y no se asume que represente un año completo. Las solicitudes abiertas no se imputan como resueltas.

El P75 se define como una regla descriptiva global del rango consultado antes del experimento predictivo. En una implementación prospectiva, el umbral debería calcularse exclusivamente con el histórico disponible antes del período que se desea predecir.

Para reproducir el análisis, ejecute las celdas en orden con SQL Server accesible. Use `RUN_GRID_SEARCH=True` para repetir los 54 ajustes y `False` para una demostración con los parámetros ya validados. La última sección valida el esquema, carga un staging temporal y publica perfiles en una transacción.


## 22 Persistencia reproducible en `bi.PerfilCluster`

La tabla destino se inspecciona en SQL Server antes de escribir. Los perfiles se insertan primero en `#PerfilClusterStage`, se validan la terna `agency + complaint_type + borough`, los rangos y la metadata del modelo, y solo después se reemplaza el contenido de `bi.PerfilCluster` dentro de una transacción. Si falla una validación o el `INSERT`, se ejecuta `ROLLBACK`; no se usan CSV ni filas escritas a mano.


In [ ]:
publish_columns = [
    'agency', 'complaint_type', 'borough', 'cluster', 'cluster_nombre',
    'solicitudes', 'resueltas', 'mediana_resolucion', 'promedio_resolucion',
    'porcentaje_demora', 'porcentaje_sin_resolucion',
]
profiles_to_publish = profiles[publish_columns].copy()
profiles_to_publish = profiles_to_publish.rename(columns={'cluster': 'cluster_id_original'})
profiles_to_publish['cluster_id_original'] = profiles_to_publish['cluster_id_original'].astype('int64')
profiles_to_publish['solicitudes'] = profiles_to_publish['solicitudes'].astype('int64')
profiles_to_publish['resueltas'] = profiles_to_publish['resueltas'].astype('int64')
profiles_to_publish['mediana_resolucion'] = profiles_to_publish['mediana_resolucion'].round(2)
profiles_to_publish['promedio_resolucion'] = profiles_to_publish['promedio_resolucion'].round(2)
profiles_to_publish['porcentaje_demora'] = profiles_to_publish['porcentaje_demora'].round(8)
profiles_to_publish['porcentaje_sin_resolucion'] = profiles_to_publish['porcentaje_sin_resolucion'].round(8)
profiles_to_publish['modelo_version'] = MODEL_VERSION
profiles_to_publish['periodo_inicio'] = period_start
profiles_to_publish['periodo_fin'] = period_end

profile_key = ['agency', 'complaint_type', 'borough']
if profiles_to_publish.duplicated(profile_key, keep=False).any():
    raise ValueError('K-Means produjo más de una fila para una terna de perfil.')
if profiles_to_publish[['agency', 'complaint_type']].isna().any().any():
    raise ValueError('La tabla destino no permite agency/complaint_type nulos.')
if profiles_to_publish['cluster_id_original'].nunique() != 3:
    raise ValueError('Se esperaban exactamente tres clusters K-Means.')
if (profiles_to_publish['solicitudes'] <= 0).any():
    raise ValueError('Hay perfiles con solicitudes no positivas.')
if (profiles_to_publish['resueltas'] > profiles_to_publish['solicitudes']).any():
    raise ValueError('Hay perfiles con resueltas mayores que solicitudes.')
if not profiles_to_publish['porcentaje_demora'].between(0, 1).all() or not profiles_to_publish['porcentaje_sin_resolucion'].between(0, 1).all():
    raise ValueError('Los porcentajes del perfil están fuera de [0, 1].')

def _native_sql_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value

def _open_publish_connection():
    parts = [
        f'DRIVER={{{driver}}}',
        f'SERVER={SQL_SERVER}',
        f'DATABASE={SQL_DATABASE}',
        'TrustServerCertificate=yes',
        'Encrypt=no',
    ]
    if credentials:
        parts.extend([f'UID={credentials["username"]}', f'PWD={credentials["password"]}'])
    else:
        parts.append('Trusted_Connection=yes')
    return pyodbc.connect(';'.join(parts), timeout=30, autocommit=False)

expected_profile_columns = {
    'agency', 'complaint_type', 'borough', 'cluster_id_original', 'cluster_nombre',
    'solicitudes', 'resueltas', 'mediana_resolucion', 'promedio_resolucion',
    'porcentaje_demora', 'porcentaje_sin_resolucion', 'modelo_version',
    'periodo_inicio', 'periodo_fin',
}
publish_order = [
    'agency', 'complaint_type', 'borough', 'cluster_id_original', 'cluster_nombre',
    'solicitudes', 'resueltas', 'mediana_resolucion', 'promedio_resolucion',
    'porcentaje_demora', 'porcentaje_sin_resolucion', 'modelo_version',
    'periodo_inicio', 'periodo_fin',
]
profile_rows = [
    tuple(_native_sql_value(value) for value in row)
    for row in profiles_to_publish[publish_order].itertuples(index=False, name=None)
]

with _open_publish_connection() as publish_connection:
    publish_cursor = publish_connection.cursor()
    schema_rows = publish_cursor.execute('''
        SELECT c.name, t.name AS type_name, c.max_length, c.precision, c.scale,
               c.is_nullable, c.is_identity
        FROM sys.columns AS c
        JOIN sys.types AS t ON t.user_type_id = c.user_type_id
        WHERE c.object_id = OBJECT_ID(N'bi.PerfilCluster', N'U')
        ORDER BY c.column_id;
    ''').fetchall()
    actual_profile_columns = {row[0] for row in schema_rows}
    missing_profile_columns = expected_profile_columns - actual_profile_columns
    if not schema_rows or missing_profile_columns:
        raise RuntimeError(f'bi.PerfilCluster no cumple el esquema esperado. Faltan: {sorted(missing_profile_columns)}')
    identity_columns = {row[0] for row in schema_rows if row[6]}
    if 'perfil_cluster_key' not in identity_columns:
        raise RuntimeError('bi.PerfilCluster no tiene la clave identity esperada; se cancela la publicación.')
    permissions = publish_cursor.execute('''
        SELECT HAS_PERMS_BY_NAME(N'bi.PerfilCluster', 'OBJECT', 'SELECT'),
               HAS_PERMS_BY_NAME(N'bi.PerfilCluster', 'OBJECT', 'INSERT'),
               HAS_PERMS_BY_NAME(N'bi.PerfilCluster', 'OBJECT', 'DELETE')
    ''').fetchone()
    if permissions is None or any(int(permission or 0) != 1 for permission in permissions):
        raise PermissionError('La cuenta no tiene SELECT/INSERT/DELETE sobre bi.PerfilCluster; no se escribió nada.')

    publish_cursor.execute('''
        SELECT TOP (0)
            agency, complaint_type, borough, cluster_id_original, cluster_nombre,
            solicitudes, resueltas, mediana_resolucion, promedio_resolucion,
            porcentaje_demora, porcentaje_sin_resolucion, modelo_version,
            periodo_inicio, periodo_fin
        INTO #PerfilClusterStage
        FROM bi.PerfilCluster;
    ''')
    stage_insert = '''
        INSERT INTO #PerfilClusterStage
        (agency, complaint_type, borough, cluster_id_original, cluster_nombre,
         solicitudes, resueltas, mediana_resolucion, promedio_resolucion,
         porcentaje_demora, porcentaje_sin_resolucion, modelo_version,
         periodo_inicio, periodo_fin)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    '''
    # Son solo perfiles (no tickets); executemany normal evita que ODBC 18
    # infiera un tipo numérico desde la primera fila y falle con fechas/texto.
    publish_cursor.fast_executemany = False
    publish_cursor.executemany(stage_insert, profile_rows)
    stage_count = int(publish_cursor.execute('SELECT COUNT_BIG(*) FROM #PerfilClusterStage').fetchone()[0])
    if stage_count != len(profile_rows):
        raise RuntimeError(f'Staging incompleto: se esperaban {len(profile_rows)} filas y llegaron {stage_count}.')
    duplicate_stage = publish_cursor.execute('''
        SELECT TOP (1) agency, complaint_type, borough
        FROM #PerfilClusterStage
        GROUP BY agency, complaint_type, borough
        HAVING COUNT_BIG(*) > 1;
    ''').fetchone()
    if duplicate_stage:
        raise RuntimeError(f'Staging duplicado para la terna {duplicate_stage}.')
    invalid_stage = publish_cursor.execute('''
        SELECT COUNT_BIG(*)
        FROM #PerfilClusterStage
        WHERE solicitudes <= 0
           OR resueltas < 0 OR resueltas > solicitudes
           OR mediana_resolucion IS NULL
           OR porcentaje_demora NOT BETWEEN 0 AND 1
           OR porcentaje_sin_resolucion NOT BETWEEN 0 AND 1
           OR modelo_version IS NULL OR periodo_inicio IS NULL OR periodo_fin IS NULL;
    ''').fetchone()[0]
    if int(invalid_stage) != 0:
        raise RuntimeError(f'Staging con {int(invalid_stage)} filas inválidas.')
    publish_connection.commit()

    try:
        publish_cursor.execute('BEGIN TRANSACTION;')
        publish_cursor.execute('DELETE FROM bi.PerfilCluster;')
        publish_cursor.execute('''
            INSERT INTO bi.PerfilCluster
            (agency, complaint_type, borough, cluster_id_original, cluster_nombre,
             solicitudes, resueltas, mediana_resolucion, promedio_resolucion,
             porcentaje_demora, porcentaje_sin_resolucion, modelo_version,
             periodo_inicio, periodo_fin)
            SELECT agency, complaint_type, borough, cluster_id_original, cluster_nombre,
                   solicitudes, resueltas, mediana_resolucion, promedio_resolucion,
                   porcentaje_demora, porcentaje_sin_resolucion, modelo_version,
                   periodo_inicio, periodo_fin
            FROM #PerfilClusterStage;
        ''')
        target_count = int(publish_cursor.execute('SELECT COUNT_BIG(*) FROM bi.PerfilCluster').fetchone()[0])
        duplicate_target = publish_cursor.execute('''
            SELECT TOP (1) agency, complaint_type, borough
            FROM bi.PerfilCluster
            GROUP BY agency, complaint_type, borough
            HAVING COUNT_BIG(*) > 1;
        ''').fetchone()
        if target_count != stage_count or duplicate_target:
            raise RuntimeError(f'Validación de destino falló: filas={target_count}, duplicado={duplicate_target}.')
        publish_cursor.execute('COMMIT TRANSACTION;')
    except Exception:
        publish_connection.rollback()
        raise

    view_check = publish_cursor.execute('''
        SELECT COUNT_BIG(*), COUNT_BIG(DISTINCT codigo_solicitud),
               SUM(CASE WHEN es_segmentado = 1 THEN 1 ELSE 0 END),
               SUM(CASE WHEN es_segmentado = 0 THEN 1 ELSE 0 END)
        FROM bi.vw_solicitudes_segmentadas;
    ''').fetchone()
    view_rows, view_distinct, segmented_rows, unsegmented_rows = (int(value or 0) for value in view_check)
    if view_rows != len(df) or view_distinct != len(df) or segmented_rows + unsegmented_rows != view_rows:
        raise RuntimeError(f'La vista no reconcilia el grano: filas={view_rows}, distintos={view_distinct}, segmentadas={segmented_rows}, fuera={unsegmented_rows}, fuente={len(df)}.')

publication_result = {
    'perfiles_publicados': target_count,
    'solicitudes_fuente': len(df),
    'solicitudes_vista': view_rows,
    'solicitudes_distintas_vista': view_distinct,
    'segmentadas': segmented_rows,
    'fuera_de_perfil': unsegmented_rows,
    'modelo_version': MODEL_VERSION,
    'periodo': period_label,
}
print(f'Publicación confirmada: {publication_result}')